# Vision-to-Text Transformer (From Scratch)
This notebook implements a pure PyTorch Transformer configuration, substituting the LSTM sequence loop entirely for parallel Self-Attention processing.

In [1]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q

import os
import pandas as pd
import spacy
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from tqdm.notebook import tqdm
import nltk
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
from collections import defaultdict
import numpy as np
import gc
import math

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
spacy_eng = spacy.load("en_core_web_sm")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Using device: cuda


In [2]:
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer_eng(text):
        return [tok.text.lower() for tok in spacy_eng.tokenizer(str(text))]

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer_eng(sentence):
                if word not in frequencies:
                    frequencies[word] = 1
                else:
                    frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

class FlickrDataset(Dataset):
    def __init__(self, root_dir, df, transform=None, vocab=None):
        self.root_dir = root_dir
        self.df = df
        
        if 'image' in self.df.columns:
            self.imgs = self.df["image"].values
        elif 'image_name' in self.df.columns:
            self.imgs = self.df["image_name"].values
            
        if 'caption' in self.df.columns:
            self.captions = self.df["caption"].astype(str).values
        elif 'comment' in self.df.columns:
            self.captions = self.df["comment"].astype(str).values
            
        self.transform = transform
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index].strip()
        img_path = os.path.join(self.root_dir, img_id)
        
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption.append(self.vocab.stoi["<EOS>"])

        return img, torch.tensor(numericalized_caption), img_id

class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        targets = [item[1] for item in batch]
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)
        img_ids = [item[2] for item in batch]
        return imgs, targets, img_ids

def prepare_datasets(root_folder, annotation_file, transform, freq_threshold=5):
    df = pd.read_csv(annotation_file, sep=',', engine='python', on_bad_lines='skip')
    df.columns = df.columns.str.strip().str.lower()
    
    img_col = 'image' if 'image' in df.columns else 'image_name'
    cap_col = 'caption' if 'caption' in df.columns else 'comment'
    
    unique_imgs = df[img_col].unique()
    train_imgs = set(unique_imgs[:29000])
    val_imgs = set(unique_imgs[29000:30000])
    test_imgs = set(unique_imgs[30000:31000]) 
    
    train_df = df[df[img_col].isin(train_imgs)]
    val_df = df[df[img_col].isin(val_imgs)]
    test_df = df[df[img_col].isin(test_imgs)]
    
    vocab = Vocabulary(freq_threshold)
    vocab.build_vocabulary(train_df[cap_col].astype(str).tolist())
    
    train_dataset = FlickrDataset(root_folder, train_df, transform=transform, vocab=vocab)
    val_dataset = FlickrDataset(root_folder, val_df, transform=transform, vocab=vocab)
    test_dataset = FlickrDataset(root_folder, test_df, transform=transform, vocab=vocab)
    
    return train_dataset, val_dataset, test_dataset

In [3]:
class EncoderCNN_VGG_Transformer(nn.Module):
    def __init__(self, encoded_image_size=14, embed_dim=512):
        super(EncoderCNN_VGG_Transformer, self).__init__()
        self.enc_image_size = encoded_image_size
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        modules = list(vgg.features.children())[:-1] 
        self.vgg = nn.Sequential(*modules)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))
        
        # Transformer mathematically needs spatial positional embedding
        self.spatial_pos_emb = nn.Embedding(encoded_image_size * encoded_image_size, embed_dim)

    def forward(self, images):
        features = self.vgg(images) 
        features = self.adaptive_pool(features) # [batch_size, 512, 14, 14]
        features = features.view(features.size(0), features.size(1), -1).permute(0, 2, 1) 
        # features: [batch_size, 196, 512]
        
        positions = torch.arange(0, self.enc_image_size * self.enc_image_size).to(features.device)
        spatial_emb = self.spatial_pos_emb(positions) # [196, 512]
        features = features + spatial_emb.unsqueeze(0)
        
        return features

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) 
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerDecoderArchitecture(nn.Module):
    def __init__(self, embed_dim, vocab_size, num_layers=3, nhead=8, dropout=0.1, max_seq_len=500):
        super(TransformerDecoderArchitecture, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = PositionalEncoding(embed_dim, max_len=max_seq_len)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=nhead, dropout=dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)
    
    def generate_square_subsequent_mask(self, sz):
        # En PyTorch >= 2.0, les masques de causalit� et de padding DOIVENT avoir le m�me type (boolean)
        # True signifie "Cacher/Ignorer" le mot futur
        return torch.triu(torch.ones(sz, sz, dtype=torch.bool), diagonal=1)

    def forward(self, features, captions, pad_idx):
        # Scale embeddings roughly by sqrt per norm
        tgt = self.embedding(captions) * math.sqrt(self.embedding.embedding_dim) 
        tgt = self.pos_encoder(tgt)
        
        seq_len = tgt.size(1)
        tgt_mask = self.generate_square_subsequent_mask(seq_len).to(tgt.device)
        
        tgt_key_padding_mask = (captions == pad_idx).to(tgt.device)
        
        out = self.transformer_decoder(
            tgt=tgt, 
            memory=features, 
            tgt_mask=tgt_mask, 
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        
        preds = self.fc_out(out)
        return preds

In [4]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

root_folder = "/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images/" 
annotation_file = "/kaggle/input/datasets/eeshawn/flickr30k/captions.txt"

embed_size = 512 
learning_rate = 3e-4 # Transformers prefer slightly smaller default LR
num_epochs = 15
batch_size = 64 

if os.path.exists(root_folder):
    train_ds, val_ds, test_ds = prepare_datasets(root_folder, annotation_file, transform)
    
    pad_idx = train_ds.vocab.stoi["<PAD>"]
    train_loader = DataLoader(dataset=train_ds, batch_size=batch_size, num_workers=2, 
                        shuffle=True, collate_fn=MyCollate(pad_idx=pad_idx), pin_memory=True)
    
    test_loader = DataLoader(dataset=test_ds, batch_size=1, num_workers=2, 
                        shuffle=False, collate_fn=MyCollate(pad_idx=pad_idx), pin_memory=True)
    
    vocab_size = len(train_ds.vocab)
    
    encoder = EncoderCNN_VGG_Transformer(embed_dim=embed_size).to(device)
    decoder = TransformerDecoderArchitecture(embed_dim=embed_size, vocab_size=vocab_size).to(device)
    
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    
    for param in encoder.vgg.parameters():
        param.requires_grad = False
        
    # Spatial Embeddings will be optimized along with the decoder
    all_params = list(decoder.parameters()) + list(encoder.spatial_pos_emb.parameters())
    optimizer = optim.Adam(all_params, lr=learning_rate)

    torch.cuda.empty_cache()
    gc.collect()
else:
    encoder = None
    decoder = None

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 225MB/s]


In [5]:
image_to_captions_test = defaultdict(list)
if encoder:
    for i in range(len(test_ds.df)):
        img_name = test_ds.df.iloc[i].image if 'image' in test_ds.df.columns else test_ds.df.iloc[i].image_name
        cap = test_ds.df.iloc[i].caption if 'caption' in test_ds.df.columns else test_ds.df.iloc[i].comment
        num_caps = [test_ds.vocab.stoi["<SOS>"]] + test_ds.vocab.numericalize(str(cap)) + [test_ds.vocab.stoi["<EOS>"]]
        str_cap = [test_ds.vocab.itos[idx] for idx in num_caps if idx not in [test_ds.vocab.stoi["<SOS>"], test_ds.vocab.stoi["<EOS>"], pad_idx]]
        image_to_captions_test[img_name].append(str_cap)

In [6]:
if decoder:
    for epoch in range(num_epochs):
        encoder.eval() 
        decoder.train()
        epoch_loss = 0
        
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}")
        for idx, (imgs, captions, img_ids) in pbar:
            imgs = imgs.to(device)
            captions = captions.to(device) 
            
            # Transformer Parallel magic happens here:
            # We predict the sequence 'shifted' by one mathematically.
            # tgt_input contains SOS but no EOS (seq_len - 1)
            # targets contains EOS but no SOS (shifted target)
            tgt_input = captions[:, :-1]
            targets = captions[:, 1:]
            
            with torch.no_grad():
                features = encoder.vgg(imgs)
                features = encoder.adaptive_pool(features) 
                features = features.view(features.size(0), features.size(1), -1).permute(0, 2, 1) 
            
            # Add positional encodings manually to bypass the `torch.no_grad` restriction
            positions = torch.arange(0, encoder.enc_image_size * encoder.enc_image_size).to(features.device)
            spatial_emb = encoder.spatial_pos_emb(positions) 
            features = features + spatial_emb.unsqueeze(0)
            
            # Forward calculates ALL words natively and concurrently
            outputs = decoder(features, tgt_input, pad_idx)
            
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            
            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=5)
            optimizer.step()
            
            epoch_loss += loss.item() 
            
            if idx % 100 == 0:
                pbar.set_postfix({"Loss": loss.item()})
                
        avg_loss = epoch_loss/len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}] | Average Training Loss: {avg_loss:.4f}")
        torch.save({'decoder': decoder.state_dict(), 'enc_pos': encoder.spatial_pos_emb.state_dict()}, f"transformer_model_ep{epoch+1}.pth")

Epoch 1/15:   0%|          | 0/2266 [00:00<?, ?it/s]

Epoch [1/15] | Average Training Loss: 3.2378


Epoch 2/15:   0%|          | 0/2266 [00:00<?, ?it/s]

Epoch [2/15] | Average Training Loss: 2.7182


Epoch 3/15:   0%|          | 0/2266 [00:00<?, ?it/s]

...

Epoch [13/15] | Average Training Loss: 1.8410


Epoch 14/15:   0%|          | 0/2266 [00:00<?, ?it/s]

Epoch [14/15] | Average Training Loss: 1.8058


Epoch 15/15:   0%|          | 0/2266 [00:00<?, ?it/s]

Epoch [15/15] | Average Training Loss: 1.7734


## Evaluation: Auto-regressive Inference

In [7]:
def evaluate_scores_strict(encoder, decoder, loader, dataset):
    encoder.eval()
    decoder.eval()
    references_bleu = []
    hypotheses = []
    
    print("Generating strictly on Test set with Auto-regressive Loop...")
    with torch.no_grad():
        for idx, (imgs, captions, img_ids) in tqdm(enumerate(loader), total=len(loader)):
            img = imgs.to(device)
            img_name = img_ids[0]
            
            # Calculate memory (features) only once!
            features = encoder(img)
            
            # Start with only '<SOS>'
            word_seq = torch.tensor([[dataset.vocab.stoi["<SOS>"]]]).to(device)
            pred_indices = []
            
            for _ in range(30):
                # We continuously feed the generated string into the Transformer
                preds = decoder(features, word_seq, pad_idx)
                
                # Snatch the last token probability matrix created mathematically
                predicted = preds[:, -1, :].argmax(1) 
                
                pred_indices.append(predicted.item())
                
                if dataset.vocab.itos[predicted.item()] == "<EOS>":
                    break
                    
                # Append the exact word to the sequence context
                word_seq = torch.cat((word_seq, predicted.unsqueeze(1)), dim=1)
            
            pred_words = [dataset.vocab.itos[idx] for idx in pred_indices if idx not in [dataset.vocab.stoi["<SOS>"], dataset.vocab.stoi["<EOS>"]]]
            
            if img_name in image_to_captions_test:
                references_bleu.append(image_to_captions_test[img_name])
                hypotheses.append(pred_words)
                del image_to_captions_test[img_name]

    print(f"Evaluated distinctly on {len(hypotheses)} test images.")
    # BLEU
    print(f"BLEU-1: {corpus_bleu(references_bleu, hypotheses, weights=(1.0, 0, 0, 0))*100:.1f}")
    print(f"BLEU-2: {corpus_bleu(references_bleu, hypotheses, weights=(0.5, 0.5, 0, 0))*100:.1f}")
    print(f"BLEU-3: {corpus_bleu(references_bleu, hypotheses, weights=(0.33, 0.33, 0.33, 0))*100:.1f}")
    print(f"BLEU-4: {corpus_bleu(references_bleu, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))*100:.1f}")
    
    # METEOR calculation
    meteor_scores = []
    for refs, hyp in zip(references_bleu, hypotheses):
        score = meteor_score(refs, hyp)
        meteor_scores.append(score)
        
    print(f"METEOR: {np.mean(meteor_scores)*100:.2f}")

if decoder:
    evaluate_scores_strict(encoder, decoder, test_loader, test_ds)

Generating strictly on Test set with Auto-regressive Loop...


  0%|          | 0/5000 [00:00<?, ?it/s]

Evaluated distinctly on 1000 test images.
BLEU-1: 62.8
BLEU-2: 40.4
BLEU-3: 26.6
BLEU-4: 17.5
METEOR: 37.03


In [8]:
import pickle

if decoder:
    checkpoint = {
        'encoder_spatial_pos_emb': encoder.spatial_pos_emb.state_dict(),
        'decoder_state_dict': decoder.state_dict(),
        'vocab_stoi': train_ds.vocab.stoi,
        'vocab_itos': train_ds.vocab.itos,
        'embed_size': embed_size,
    }

    # Sauvegarder sous format .pkl
    file_name = 'image_captioning_transformer_model.pkl'
    torch.save(checkpoint, file_name)

    print(f"Model checkpoint complete: '{file_name}'")

Model checkpoint complete: 'image_captioning_transformer_model.pkl'
